# Step 8 - Robustness Analysis (Sensor Perturbations)

Inject physically motivated perturbations into test-set sensor readings (temperature +1/+2C, humidity +-5%, CO2 +-20 ppm, light +-20 Lux, and combined Gaussian noise) to emulate calibration drift and noisy IoT hardware. Models are *not* retrained; we measure performance degradation.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def _find_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "data").is_dir():
            return cand
    return p

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as C
from src import viz
viz.setup_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Repository root:", ROOT)


Repository root: /Users/rauankaztaev/IdeaProjects/Draft/project


In [2]:
import joblib
from src import data, models, benchmark, robustness
from src import utils

df = data.load_clean()
x_train, x_test, y_train, y_test = data.get_splits(df)
store_path = C.MODELS_DIR / "test_predictions.joblib"
NAMES = ["Random Forest", "Extra Trees", "XGBoost", "LightGBM", "CatBoost",
         "Gradient Boosting", "Logistic Regression", "SVM", "KNN", "Decision Tree"]
if store_path.exists():
    fitted = {n: utils.load_model(n) for n in NAMES if (C.MODELS_DIR / f"{n.lower().replace(' ','_')}.joblib").exists()}
else:
    zoo = models.get_model_zoo(x_train)
    _, fitted, _ = benchmark.evaluate_on_test(x_train, y_train, x_test, y_test, zoo)
NAMES = [n for n in NAMES if n in fitted]
print("Evaluating robustness for:", NAMES)

Evaluating robustness for: ['Random Forest', 'Extra Trees', 'XGBoost', 'LightGBM', 'CatBoost', 'Gradient Boosting', 'Logistic Regression', 'SVM', 'KNN', 'Decision Tree']


In [3]:
long_df = robustness.robustness_analysis(fitted, x_test, y_test, NAMES)
summary = robustness.robustness_summary(long_df)
display(summary)
utils.save_table(long_df.round(4), "robustness_long")
utils.save_table(summary, "robustness_summary",
                 caption="Mean F1 degradation under sensor perturbations.", label="tab:robust")

,Model,mean_F1_under_perturbation,worst_F1,mean_F1_drop,max_F1_drop
0,XGBoost,0.8465,0.5344,0.1514,0.4635
1,Extra Trees,0.8462,0.6194,0.1517,0.3785
2,Gradient Boosting,0.8364,0.4949,0.1587,0.5001
3,CatBoost,0.8322,0.5306,0.1657,0.4673
4,Random Forest,0.8295,0.6154,0.1670,0.3811
5,KNN,0.8215,0.4351,0.1735,0.5600
6,Decision Tree,0.8028,0.4605,0.1944,0.5367
7,LightGBM,0.7959,0.4605,0.2005,0.5360
8,SVM,0.7868,0.4415,0.2034,0.5487
9,Logistic Regression,0.6657,0.2363,0.2352,0.6646


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/robustness_summary.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/robustness_summary.tex')}

## 6.1 Robustness heatmap and degradation ranking

In [4]:
import seaborn as sns
pivot = long_df.pivot(index="Model", columns="Scenario", values="F1")
order = summary["Model"].tolist()
cols = ["Clean"] + [c for c in pivot.columns if c != "Clean"]
pivot = pivot.loc[order, cols]
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=1.0, ax=ax)
ax.set_title("F1 under sensor perturbations")
fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "robustness_heatmap.png", dpi=300, bbox_inches="tight"); plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4.5))
s = summary.sort_values("mean_F1_drop")
sns.barplot(x="mean_F1_drop", y="Model", data=s, ax=ax, hue="Model", legend=False)
ax.set_title("Mean F1 drop under perturbation (lower = more robust)")
fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "robustness_ranking.png", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Most robust:", s.iloc[0]["Model"], "| Least robust:", s.iloc[-1]["Model"])

Most robust: XGBoost | Least robust: Logistic Regression


**Interpretation.** The most robust model retains the highest F1 under the harshest perturbations. Because `Light` is the dominant feature, light-sensor drift causes the largest degradation. Practically, this means field deployments should prioritise light-sensor calibration, and the recommended model balances peak accuracy against graceful degradation, not accuracy alone.